### Importing the needed libraries

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Reading the data

In [2]:
df = pd.read_csv("../_output/processed_dogs.csv")
print("Shape of the dataframe:")
print(df.shape)
df.head()

Shape of the dataframe:
(2920, 19)


,name,age,sex,breed,date_found,adoptable_from,posted,color,coat,size,neutered,housebroken,likes_people,likes_children,get_along_males,get_along_females,get_along_cats,keep_in,found_year
0,Gida,0.25,female,Unknown Mix,2019-12-10,2019-12-11,2019-12-11,red,short,small,no,unknown,unknown,unknown,unknown,unknown,unknown,unknown,2019
1,Frida És Ricsi,0.17,female,Unknown Mix,2019-12-01,2019-12-01,2019-12-09,black and white,short,small,no,unknown,yes,yes,yes,yes,yes,unknown,2019
2,Unknown,4.00,male,Unknown Mix,2019-12-08,2019-12-23,2019-12-08,saddle back,short,medium,no,unknown,unknown,unknown,unknown,unknown,unknown,unknown,2019
3,Unknown,1.00,male,Unknown Mix,2019-12-08,2019-12-23,2019-12-08,yellow-brown,medium,medium,no,unknown,unknown,unknown,unknown,unknown,unknown,unknown,2019
4,Amy,2.00,female,French Bulldog Mix,2019-12-10,2019-12-11,2019-12-11,black,short,small,no,unknown,unknown,unknown,unknown,unknown,unknown,unknown,2019


##### Some processing and feature engineering before moving to the main plots and insights (Story)

In [3]:
# Constant variables for the notebook
DATE_COLS = ["date_found", "adoptable_from", "posted"]
BEHAVIOR_COLS = ["likes_people", "likes_children", "get_along_males", "get_along_females", "get_along_cats"]
AGE_GROUPS = ['Young (0-5y)', 'Adult (5-10y)', 'Senior (10+y)']

In [4]:
# As we are going to analyzae the behavior of dogs, we need to filter out the dogs that have a lot of unknown behavior
# By the way a lot of unknowns in the dataset
# 
threshold = len(BEHAVIOR_COLS) - 1
df = df[(df[BEHAVIOR_COLS] == "unknown").sum(axis=1) < threshold]

In [5]:
# Changing datetime columns to a proper format
for col in DATE_COLS:
    df[col] = pd.to_datetime(df[col])

# Creating a new column to calculate the waiting days between date_found and adoptable_from
df["waiting_days"] = (df["adoptable_from"] - df["date_found"]).dt.days
df = df[df['waiting_days'] >= 0]
# If the dog is delayed or not at all
df["delayed"] = (df["waiting_days"] > 0).astype(int)

Here we saw that most of the dogs ready to be adopted after 0 days. This makes comparing by days and other variable useless. That is why we will either look at them by filtering only more then 0 days or by looking at the mean values.

In [6]:
# Creating a behavior score column which will show overall behavior of the specific dog
# If the has 'yes' for the behavioral column it scores go up by 1 point
df['behavior_score'] = df[BEHAVIOR_COLS].apply(lambda row: sum(row == 'yes'), axis=1)
df['behavior_score_name'] = 'Score_' + df['behavior_score'].astype(str)

# Categorizing dogs based on their behavioral score
def classify_behavior(row):
    traits = row[BEHAVIOR_COLS]
    positive = (traits == 'yes').sum()
    if positive >= 4:
        return 'Highly Social'
    elif positive == 3:
        return 'Social'
    elif positive <= 2:
        return 'Limited Social'

df['behavior_category'] = df.apply(classify_behavior, axis=1)
behavior_order = ['Limited Social', 'Social', 'Highly Social']
df['behavior_category'] = pd.Categorical(df['behavior_category'], categories=behavior_order, ordered=True)

In [7]:
df['behavioral_profile'] = (
    (df['likes_people'].astype(str) + '_' +
     df['get_along_cats'].astype(str) + '_' +
     df['neutered'].astype(str)).str.lower()
)

In [8]:
df['age_group'] = pd.cut(df['age'], 
                                  bins=[0, 5, 10, 100], 
                                  labels=AGE_GROUPS)

In [9]:
df['month_found'] = df['date_found'].dt.month
df['quarter_found'] = df['date_found'].dt.quarter
df['season_found'] = df['month_found'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
})

## Story:
##### We will try to build our story around the Delay of the adoption readiness of dogs. <br> Key Question: What drives differences in adoption readiness timelines? <br> NOTE: For descriptions and explanations of newly created columns (features) refer to the end of the notebook

In [10]:
delay_rate_size = df.groupby("size")["delayed"].mean().reset_index()
delay_rate_size["delayed_pct"] = delay_rate_size["delayed"] * 100

fig = px.bar(
    delay_rate_size,
    x="size",
    y="delayed_pct",
    title="Percentage of Dogs with Delayed Adoption by Size",
    range_y=[delay_rate_size["delayed_pct"].min() - 1,
             delay_rate_size["delayed_pct"].max() + 1],
    text=delay_rate_size["delayed_pct"].round(1),
    height=600,
    width=1100
)

fig.update_traces(textposition='outside')
fig.show()

##### Contrary to common expectations, smaller dogs appear to have a higher probability of delayed adoption readiness compared to larger dogs. This may be influenced by dataset factors such as availability, intake conditions, or reporting bias (mistakes made during data collection or annotation). Later we will look at visualizations which will show not just the binary "delayed" indicator, but the mean of delayed days.

In [11]:
behavior_size_analysis = df.groupby(['behavior_category', 'size']).agg({
    'waiting_days': ['mean', 'count'],
    'delayed': 'mean'
}).round(1)
waiting_by_behavior_size = df.groupby(['behavior_category', 'size'])['waiting_days'].mean().reset_index()
count_by_behavior_size = df.groupby(['behavior_category', 'size']).size().reset_index(name='count')
waiting_by_behavior_size = waiting_by_behavior_size.merge(count_by_behavior_size)

In [12]:
fig1 = px.scatter(
    waiting_by_behavior_size,
    x='behavior_category',
    y='size',
    size='count',
    color='waiting_days',
    hover_name='behavior_category',
    hover_data={'size': True, 'waiting_days': ':.1f', 'count': True},
    title='Adoption Readiness Timeline: Behavioral Profile vs Dog Size<br><sub>Bubble size = number of dogs; Color intensity = average waiting days',
    color_continuous_scale='RdYlGn_r',
    size_max=50,
    height=600,
    width=1100,
    labels={
        'waiting_days': 'Average Waiting Days',
        'behavior_category': 'Behavioral Category',
        'size': 'Dog Size'
    },
    category_orders={'behavior_category': ['Limited Social', 'Social', 'Highly Social']}
)

fig1.update_layout(
    font=dict(size=12),
    coloraxis_colorbar=dict(title="Avg Waiting Days")
)
fig1.show()


##### In the visualization above we can see that small and highly social dogs have the lowest waiting time until being adoption ready, which indeed makes sense because highly social small dogs are easier to handle. In contrast, large dogs have longer waiting times (small limited social dogs have the highest waiting time).

In [13]:
all_sizes = sorted(df['size'].unique())

figs_data = {}
for age_group in AGE_GROUPS:
    age_data = df[df['age_group'] == age_group]
    
    heatmap_data = age_data.pivot_table(
        values='waiting_days',
        index='behavior_category',
        columns='size',
        aggfunc='mean'
    )
    
    heatmap_data = heatmap_data.reindex(['Limited Social', 'Social', 'Highly Social'])
    heatmap_data = heatmap_data.reindex(columns=all_sizes)
    figs_data[age_group] = heatmap_data

first_age = AGE_GROUPS[0]
fig = px.imshow(
    figs_data[first_age],
    color_continuous_scale='RdYlGn_r',
    title=f'Average Waiting Days - {first_age}',
    labels=dict(x='Dog Size', y='Behavioral Profile', color='Days'),
    height=500,
    width=700,
    text_auto='.1f',
    aspect='auto'
)

fig.update_layout(
    font=dict(size=12),
    coloraxis_colorbar=dict(title="Avg Days")
)
# Buttons
buttons = []
for age_group in AGE_GROUPS:
    buttons.append(
        dict(
            label=age_group,
            method='update',
            args=[
                {'z': [figs_data[age_group].values]},
                {'title': f'Average Waiting Days - {age_group}'}
            ]
        )
    )
# Dropdown menu
fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=1.15,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="lightgray",
            bordercolor="gray",
            borderwidth=1
        )
    ]
)

fig.show()
print(f"\nBreakdown by age group:")
print(df['age_group'].value_counts().sort_index())


Breakdown by age group:
age_group
Young (0-5y)     368
Adult (5-10y)    851
Senior (10+y)    703
Name: count, dtype: int64


##### It can be clearly seen that small dogs with limited social behavior have the highest mean waiting delay before achievement of adoption readiness; however, this plot also shows that overall senior (10+ years) dogs have greater waiting delays, especially large ones. It can be deduced that senior dogs overall need more time to be prepared and treated before achieving adoption readiness.

In [14]:
df_filtered = df[df['housebroken'] != 'unknown']
print(f"With known housebroken status: {len(df_filtered)}")

analysis_data = df_filtered.groupby(['housebroken', 'size']).agg({
    'waiting_days': 'mean',
    'name': 'count'
}).reset_index()
analysis_data.columns = ['housebroken', 'size', 'avg_waiting_days', 'count']
analysis_data['formatted_days'] = analysis_data['avg_waiting_days'].apply(lambda x: f"{x:.2f}")
hb_categories = sorted(df_filtered['housebroken'].unique())

figs_hb_data = {}
for hb_option in hb_categories:
    hb_data = analysis_data[analysis_data['housebroken'] == hb_option]
    figs_hb_data[hb_option] = hb_data

first_hb = hb_categories[0]
fig = px.bar(
    figs_hb_data[first_hb],
    x='size',
    y='avg_waiting_days',
    hover_data={'count': True, 'size': True, 'avg_waiting_days': ':.2f'},
    title=f'Average Waiting Days (Housebroken: {first_hb.upper()})',
    height=500,
    width=800,
    labels={
        'size': 'Dog Size',
        'avg_waiting_days': 'Average Waiting Days',
        'count': 'Number of Dogs'
    },
    text='formatted_days'
)

fig.update_traces(textposition='outside')
fig.update_layout(font=dict(size=12))

# Buttons
buttons = []
for hb_option in hb_categories:
    buttons.append(
        dict(
            label=f"Housebroken: {hb_option.upper()}",
            method='update',
            args=[
                {'x': [figs_hb_data[hb_option]['size']],
                 'y': [figs_hb_data[hb_option]['avg_waiting_days']],
                 'text': [figs_hb_data[hb_option]['formatted_days']]},
                {'title': f'Average Waiting Days (Housebroken: {hb_option.upper()})'}
            ]
        )
    )
# Dropdown Menu
fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.15,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="lightgray",
            bordercolor="gray",
            borderwidth=1
        )
    ]
)

fig.show()

With known housebroken status: 421


##### When selecting the housebroken option as `Yes`, it becomes clear that small housebroken dogs have, on average, less delay in achieving adoption readiness. For the housebroken option `No`, the numbers are relatively small; therefore, we focus only on housebroken dogs. <br> NOTE: Even though this visualization reveals patterns, the absolute number of dogs with known housebroken status is relatively low.

In [15]:
temporal_analysis = df.groupby(['season_found', 'size']).agg({
    'waiting_days': 'mean',
    'delayed': 'mean',
    'name': 'count'
}).reset_index()

temporal_analysis.columns = ['season', 'size', 'avg_waiting', 'delay_rate', 'count']

size_options = sorted(temporal_analysis['size'].unique())
figs_season_data = {}

for size_opt in size_options:
    size_data = temporal_analysis[temporal_analysis['size'] == size_opt].sort_values('season')
    figs_season_data[size_opt] = size_data

first_size = size_options[0]
fig = px.bar(
    figs_season_data[first_size],
    x='season',
    y='delay_rate',
    hover_data={'count': True, 'delay_rate': ':.1%', 'avg_waiting': ':.1f'},
    title=f'Delay Rate by Season (Size: {first_size.upper()})',
    height=600,
    width=900,
    labels={
        'delay_rate': 'Delay Rate (%)',
        'season': 'Season Found',
        'size': 'Dog Size',
        'count': 'Number of Dogs',
        'avg_waiting': 'Average Waiting Days'
    }
)

fig.update_traces(textposition='outside', text=figs_season_data[first_size]['delay_rate'].apply(lambda x: f"{x*100:.1f}%"))
fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12)
)
# Buttons
buttons = []
for size_opt in size_options:
    buttons.append(
        dict(
            label=f"Size: {size_opt.upper()}",
            method='update',
            args=[
                {'x': [figs_season_data[size_opt]['season']],
                 'y': [figs_season_data[size_opt]['delay_rate']],
                 'text': [figs_season_data[size_opt]['delay_rate'].apply(lambda x: f"{x*100:.1f}%")]},
                {'title': f'Delay Rate by Season (Size: {size_opt.upper()})'}
            ]
        )
    )
# Dropdown Menu
fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.99,
            xanchor="right",
            y=1.1,
            yanchor="top",
            bgcolor="lightgray",
            bordercolor="gray",
            borderwidth=1
        )
    ]
)
fig.show()

##### This visualization tests the hypothesis that delay rate may also depend on the season when the dog was found. Initially, we hypothesized that winter would have the highest delay rate; however, summer actually shows the highest rate across all dog sizes. We hypothesize that during summer, dogs require more care such as hair grooming and vaccinations against certain parasites and insects.

## Conclusion

The analysis story reveals that adoption readiness is influenced by multiple of variables. 

##### Key takeaways:

- Highly social dogs, regardless of size, achieve adoption readiness faster than limited social dogs
- While counterintuitive, smaller dogs face longer delays, but this is mediated by their behavioral profile
- Senior dogs consistently require more preparation time, suggesting tailored care protocols for older animals
- Housebroken dogs, especially smaller breeds, move through the adoption pipeline significantly faster
- Summer peaks in delay rates may indicate increased care needs due to health and grooming requirements

#### Engineered Features (columns) descriptions

| Column | Description |
|--------|-------------|
| **waiting_days** | Number of days between date_found and adoptable_from (adoption readiness delay) |
| **delayed** | Binary indicator: 1 if waiting_days > 0, else 0 |
| **behavior_score** | Count of 'yes' values across all behavior columns (0-5 range) |
| **behavior_score_name** | String representation of behavior_score (e.g., 'Score_3') |
| **behavior_category** | Categorical classification based on behavior_score: 'Limited Social' (≤2 yes), 'Social' (3 yes), or 'Highly Social' (≥4 yes) |
| **behavioral_profile** | String combining likes_people, get_along_cats, and neutered status |
| **age_group** | Age categorization: 'Young (0-5y)', 'Adult (5-10y)', or 'Senior (10+y)' |
| **month_found** | Month number (1-12) when the dog was found |
| **quarter_found** | Quarter (1-4) when the dog was found |
| **season_found** | Season when the dog was found: 'Winter', 'Spring', 'Summer', or 'Fall' |